In [1]:
# Import basic libraries for data handling
import pandas as pd
import numpy as np
import re

# Import NLP tool (TF-IDF converts text to numbers)
from sklearn.feature_extraction.text import TfidfVectorizer

# Import ML models (for comparison)
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Import evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
# Load training and testing datasets
train_df = pd.read_csv("Data_Train.csv", encoding='latin1')
test_df = pd.read_csv("Data_Test.csv", encoding='latin1')

# Display first few rows to understand structure
train_df.head()

,STORY,SECTION
0,But the most painful was the huge reversal in ...,3
1,How formidable is the opposition alliance amon...,0
2,Most Asian currencies were trading lower today...,3
3,"If you want to answer any question, click on ...",1
4,"In global markets, gold prices edged up today ...",3


In [5]:
# Check distribution of categories (0,1,2,3)
train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns = test_df.columns.str.strip().str.lower()
print(train_df['section'].value_counts())

# Check for missing values
print(train_df.isnull().sum())

section
1    2772
2    1924
0    1686
3    1246
Name: count, dtype: int64
story      0
section    0
dtype: int64


In [6]:
# Function to clean text data
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply cleaning to both train and test data
train_df['story'] = train_df['story'].apply(clean_text)
test_df['story'] = test_df['story'].apply(clean_text)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Convert text to features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

X = vectorizer.fit_transform(train_df['story'])
y = train_df['section']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data split successful")

Data split successful


In [19]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Initialize models
nb_model = MultinomialNB()
lr_model = LogisticRegression(max_iter=300)
svm_model = SVC()

# Train
nb_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)

print("Models trained")

Models trained


In [17]:
nb_pred = nb_model.predict(X_test)
lr_pred = lr_model.predict(X_test)
svm_pred = svm_model.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

evaluate(y_test, nb_pred, "Naive Bayes")
evaluate(y_test, lr_pred, "Logistic Regression")
evaluate(y_test, svm_pred, "SVM")


Naive Bayes
Accuracy: 0.963302752293578
              precision    recall  f1-score   support

           0       0.96      0.95      0.95       323
           1       0.97      0.97      0.97       549
           2       0.96      0.96      0.96       402
           3       0.95      0.98      0.97       252

    accuracy                           0.96      1526
   macro avg       0.96      0.96      0.96      1526
weighted avg       0.96      0.96      0.96      1526

[[306   7   6   4]
 [  3 530   8   8]
 [  8   8 386   0]
 [  2   2   0 248]]

Logistic Regression
Accuracy: 0.9639580602883355
              precision    recall  f1-score   support

           0       0.97      0.93      0.95       323
           1       0.97      0.98      0.97       549
           2       0.95      0.98      0.96       402
           3       0.96      0.96      0.96       252

    accuracy                           0.96      1526
   macro avg       0.96      0.96      0.96      1526
weighted avg     

In [21]:
# Based on results, choose best model
# Usually Naive Bayes works best for text
best_model = nb_model

In [22]:
# Convert numeric labels to readable categories
label_map = {
    0: "Politics",
    1: "Technology",
    2: "Entertainment",
    3: "Business"
}

In [23]:
# Function to predict category of new text
def predict_news(text):
    text = clean_text(text)  # Clean input
    vector = vectorizer.transform([text])  # Convert to TF-IDF
    pred = best_model.predict(vector)[0]  # Predict
    return label_map[pred]

# Test with sample input
print(predict_news("The government passed a new policy today"))
print(predict_news("Apple launched a new AI chip"))

Politics
Technology


In [24]:
import pickle

# Save model and vectorizer
pickle.dump(best_model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("✅ Model saved successfully!")

✅ Model saved successfully!
